# Neural Networks



## Neural Networks

&ensp; &ensp; &ensp; &ensp; **Neural networks (NNs)** are computational models inspired by the structure of the human brain, designed to recognize patterns and make predictions. They consist of layers of interconnected nodes (often called neurons) that process information through mathematical operations.

A basic neural network has the following structure:

1. **Input Layer**: It receives raw data, like pixels of an image, or tokens of a text. Each node in this layer represents an input dimension.
2. **Hidden Layers**: They are between the input layer and output layer, and process the information. Each hidden layer transforms data from the previous layer, allowing the network to progressively learn and recognize patterns.
3. **Output Layer**: It provides the network’s output.

```{figure} ../images/neural-network.png
---
width: 300px
name: neural-network
---
Basic Structure of a Neural Network
```



(2.2)=
## Neurons

&ensp; &ensp; &ensp; &ensp; A **neuron** is a unit that takes in multiple inputs and processes them to produce a single output. As shown above in [*Fig. 2.1*](neural-network), each neuron in the hidden and output layers connects to all the neurons in the previous layer. These connections have associated values, known as **weights**, which are adjusted by the model. The weights represent the strength of connection between the neurons. A greater weight indicates a stronger connection.


```{figure} ../images/neuron.png
---
width: 220px
name: neuron
---
Structure of a Neuron
```


```{admonition} Neuron's output

To calculate the output of a neuron, all the inputs to the neuron ($x_{1}, x_{2}, \dots, x_{d}$) are multiplied by their corresponding connection weights ($w_{1}, w_{2}, \dots, w_{d}$), and the products are summed. 

A numerical value, called the bias ($b$), is then added to the weighted sum. Finally, the result is passed through an **activation function** ($\sigma$) that returns the output of the neuron, the neuron's **activation value** ($h$).

<br>

$$
\small
h = \sigma\left(x_1w_1 + x_2w_2 + \dots + x_dw_d + b\right)
$$

where:

- $d$ is the number of inputs to the neuron.
```


```{important}
Please note that the ouput of a neuron is one of the inputs to all the neurons in the next layer.
```




## How NNs Learn

Neural networks learn using training data to adjust their parameters (weights and biases) to minimize the loss function. This process involves repeating these four steps during training:

1. **Forward Pass:** Process input data though network and generate predictions.
2. **Calculate Loss:** Calculate the error of the predictions.
3. **Backward Pass:** Compute gradient of the loss function with respect to the network's parameters.
4. **Update Parameters:** Adjust network's parameters to minimize the loss.




## Digits Recognizer

&ensp; &ensp; &ensp; &ensp; In this and the following chapters, we will build a discriminative model to recognize handwritten digits. To train our neural network, we will use the [MNIST dataset](https://huggingface.co/datasets/ylecun/mnist), which contains 60,000 training images and 10,000 test images. All the images are labeled, grayscale, and 28 x 28 pixels.


```{figure} ../images/digit-recognizer.png
---
width: 400px
name: digit-recognizer
---
Digit Recognizer
```

In [1]:
# Import libraries
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor

# Set print options for PyTorch tensors
torch.set_printoptions(precision=2, linewidth=50000, profile="full", sci_mode=False)

# Load the MNIST dataset
train_ds = MNIST(root="../data", train=True, download=True)

We can access the images and labels of the PyTorch Dataset object indexing

In [2]:
train_ds[0]

(<PIL.Image.Image image mode=L size=28x28>, 5)

Let's take a look at the first 2 images:

In [3]:
train_ds[0][0]

In [4]:
train_ds[1][0]

## Data Preprocessing

Please note that the images in the dataset are in PIL (Python Imaging Library) format. Typically, raw data like this is not immediately ready to be used as input for a machine learning model. The steps required to clean and prepare the data are collectively known as **data preprocessing**.

When working with PIL images in deep learning, the standard workflow involves converting the image to 8-bit format, and then transforming it into a normalized PyTorch tensor.

Suppose we have the following 4x4 grayscale PIL image:

```{figure} ../images/image-example.png
---
width: 180px
name: image-example
---
4x4 grayscale PIL image
```

When represented in 8-bit format, each pixel is stored as a single byte and takes on a value between 0 (black) and 255 (white):

```python
tensor([[   0,   42,   85,  127],
        [  42,   85,  127,  170],
        [  85,  127,  170,  213],
        [ 127,  170,  213,  255]])
```

Neural networks generally perform better when their inputs are normalized. A common normalization technique for image data is to rescale pixel values to the range [0, 1] by dividing each value by 255:

```python
tensor([[0.00, 0.16, 0.33, 0.50],
        [0.16, 0.33, 0.50, 0.67],
        [0.33, 0.50, 0.67, 0.84],
        [0.50, 0.67, 0.84, 1.00]])
```

```{admonition} Help
:class: dropdown
`ToTensor()` eficiently converts a PIL image to a normalized PyTorch tensor.
```

In [5]:
# Define the transformation
to_tensor = ToTensor()

train_x = []
train_y = []

for img, label in train_ds:
    train_x.append(to_tensor(img))  # Apply the transformation to each image
    train_y.append(label)

# Conver the lists to PyTorch tensors
train_x = torch.stack(train_x)
train_y = torch.tensor(train_y)

# Print the shapes of the resulting tensors
print(f"Train Images: {list(train_x.shape)}")
print(f"Train Labels: {list(train_y.shape)}")

Train Images: [60000, 1, 28, 28]
Train Labels: [60000]


Notice that our training images are stored as a 4D tensor of the form [N, C, H, W].

The dimensions represent:
- N – Number of images (60,000 examples in the training set).
- C – Number of channels (1 for grayscale images; 3 for RGB/color images).
- H – Image height (28 pixels).
- W – Image width (28 pixels).

In [6]:
# Print first 2 images
print(train_x[:2])

tensor([[[[0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
          [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
          [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
          [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
          [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
          [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.01, 0.07, 0.07,

Each image is currently stored as a 3D tensor of shape [1, 28, 28]. In order to process these images using a MLP, we need to flatten each one into a vector of shape [784].

```{admonition} Help
:class: dropdown
`tensor.view()` efficiently reshapes the shape of a PyTorch tensor.
```

In [7]:
# Flatten images
X = train_x.view(-1, 784) # 784 = 1 x 28 x 28

print(f"Images: {list(X.shape)}")

Images: [60000, 784]


In [8]:
# Print first 2 images
print(X[:2])

tensor([[0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.01, 0.07, 0.07, 0.07, 0.49, 0.53, 0.69, 0.10, 0.65, 1.00, 0.97, 0.50, 0.00, 0

## MLP

&ensp; &ensp; &ensp; &ensp; A **multilayer perceptron (MLP)** is a type of neural network composed of multiple layers of fully connected neurons with nonlinear activation functions. Our MLP will have 784 input neurons (one for each pixel in the image) and 10 output neurons (one for each possible class: 0 to 9).


```{figure} ../images/MLP.png
---
width: 400px
name: MLP
---
MLP
```



## Random Initialization

To keep things simple for now, we will just initialize the weights and biases of our neural network randomly.

```{admonition} Help
:class: dropdown
`torch.randn()` generates a tensor filled with random numbers drawn from a normal distribution (mean = 0, standard deviation = 1).
```

In [9]:
def initialize_nn(n_hidden = 8):

    # Seed for reproducibility
    g = torch.Generator().manual_seed(2)
     
    # Initialize parameters randomly
    W1 = torch.randn((784, n_hidden),      generator=g)
    b1 = torch.zeros(n_hidden)
    W2 = torch.randn((n_hidden, n_hidden), generator=g)
    b2 = torch.zeros(n_hidden)
    W3 = torch.randn((n_hidden, 10),       generator=g)
    b3 = torch.zeros(10)

    parameters = [W1, b1, W2, b2, W3, b3]
    
    return parameters

parameters = initialize_nn()
W1, b1, W2, b2, W3, b3 = parameters

In [10]:
print(f"W1    {list(W1.shape)}    I just printed the first 15 rows, there would be 784!")
print(W1[:15])

print(f"\nb1    {list(b1.shape)}")
print(b1)

print(f"\nW2    {list(W2.shape)}")
print(W2)

print(f"\nb2    {list(b2.shape)}")
print(b2)

print(f"\nW3    {list(W3.shape)}")
print(W3)

print(f"\nb3    {list(b3.shape)}")
print(b3)

W1    [784, 8]    I just printed the first 15 rows, there would be 784!
tensor([[-1.04,  0.92, -1.30, -1.11, -1.22,  1.17, -1.06, -0.12],
        [-0.91,  0.35, -0.57, -0.24,  1.01, -0.75, -0.22, -0.43],
        [-1.51, -0.46, -0.85,  0.53,  0.03, -0.05,  1.07,  0.89],
        [ 0.46, -0.50,  0.13,  2.76,  0.14,  1.12,  0.32,  1.75],
        [-0.76,  1.83, -0.28, -0.27, -1.29, -0.02, -0.24, -0.71],
        [ 1.16,  0.43, -1.19, -0.75, -0.93, -0.86, -0.96, -0.10],
        [-1.02,  0.32, -1.60,  1.85,  0.04,  1.59, -0.59,  1.13],
        [ 0.76, -1.20, -0.58, -0.44, -1.98,  0.78, -0.77, -0.14],
        [ 1.14, -0.64, -1.47, -0.21, -0.87,  1.62, -0.24,  0.94],
        [ 2.11, -0.98,  0.18, -0.13, -0.27,  0.34,  0.19,  2.14],
        [ 0.85,  0.10, -0.06,  0.83,  0.56, -0.78,  0.33,  0.18],
        [ 2.11,  1.07,  0.02,  1.12, -1.49, -0.20, -1.05, -1.58],
        [ 0.10, -0.35,  0.24,  0.65,  0.87, -0.93,  0.18,  1.02],
        [-0.48, -0.54, -0.53, -0.64,  0.04, -0.48, -0.04,  1.20],
    

## Forward Pass

In the **forward pass**, the input data flows through the neural network, layer by layer, to produce the network's output.


```{admonition} Neuron's output

In section [2.2 Neurons](2.2), we saw that the output of a neuron is given by the formula:

<br>

$$
\small
h = \sigma\left(x_1w_1 + x_2w_2 + \dots + x_dw_d + b\right)
$$

where:

- $d$ is the number of inputs to the neuron.
```


```{admonition} Neuron's output (dot product)

Using a dot product we can express the weighted sum as:

<br>

$$
\small
h = \sigma\left(
\begin{bmatrix}
x_1 & x_2 & \dots & x_d
\end{bmatrix}
\cdot
\begin{bmatrix}
w_1 \\
w_2 \\
\vdots \\
w_d
\end{bmatrix}
+ b\right)
$$
```


````{admonition} Layer's output (single examples)

Using a weight matrix we can calculate the activation values of all the neurons in the layer:

<br>

$$
\small
\begin{bmatrix}
h_1 & h_2 & \dots & h_m
\end{bmatrix}
=
\sigma\left(
\begin{bmatrix}
x_1 & x_2 & \dots & x_d
\end{bmatrix}
\cdot
\begin{bmatrix}
w_{11} & w_{12} & \dots & w_{1m} \\
w_{21} & w_{22} & \dots & w_{2m} \\
\vdots & \vdots & \ddots & \vdots \\
w_{d1} & w_{d2} & \dots & w_{dm} \\
\end{bmatrix}
+
\begin{bmatrix}
b_1 & b_2 & \dots & b_m
\end{bmatrix}
\right)
$$

where:

- $m$ is the number of neurons in the layer.


```{important}
Each column of the weight matrix contains the weights of the connections between a single neuron in the current layer and all the neurons in the previous layer.
```

````


````{admonition} Layer's output (multiple examples)

Using matrix multiplication we can efficiently calculate in parallel the output of a layer for several input examples:

<br>

$$
\small
\begin{bmatrix}
h_{11} & h_{12} & \dots & h_{1m} \\
h_{21} & h_{22} & \dots & h_{2m} \\
\vdots & \vdots & \ddots & \vdots \\
h_{N1} & h_{N2} & \dots & h_{Nm}
\end{bmatrix}
=
\sigma\left(
\begin{bmatrix}
x_{11} & h_{12} & \dots & h_{1d} \\
x_{21} & h_{22} & \dots & h_{2d} \\
\vdots & \vdots & \ddots & \vdots \\
x_{N1} & h_{N2} & \dots & h_{Nd}
\end{bmatrix}
\times
\begin{bmatrix}
w_{11} & w_{12} & \dots & w_{1m} \\
w_{21} & w_{22} & \dots & w_{2m} \\
\vdots & \vdots & \ddots & \vdots \\
w_{d1} & w_{d2} & \dots & w_{dm} \\
\end{bmatrix}
+
\begin{bmatrix}
b_{1} & b_{2} & \dots & b_{m}
\end{bmatrix}
\right)
$$

where:

- $N$ is the number of input examples.


```{important}
Each row of the output matrix contains the activation values of all the neurons in the layer for a single input example.
```

````




In [11]:
# forward pass
h1 = torch.tanh(X @ W1 + b1)   # (60000,  8) = (60000, 784) x (784, 8) + (8)
h2 = torch.tanh(h1 @ W2 + b2)  # (60000,  8) = (60000,   8) x (8,   8) + (8)
logits = h2 @ W3 + b3          # (60000, 10) = (60000,   8) x (8, 10)  + (10)

In [12]:
print(f"h1    {list(h1.shape)}    I just printed the first 15 rows, there would be 60000!")
print(h1[:15])

print(f"\nh2    {list(h2.shape)}    I just printed the first 15 rows, there would be 60000!")
print(W1[:15])

print(f"\nlogits    {list(logits.shape)}    I just printed the first 15 rows, there would be 60000!")
print(logits[:15])

h1    [60000, 8]    I just printed the first 15 rows, there would be 60000!
tensor([[ 1.00,  1.00, -1.00, -1.00,  1.00,  1.00,  1.00, -1.00],
        [-1.00, -0.98, -1.00, -1.00,  1.00,  1.00, -1.00, -1.00],
        [ 0.99, -1.00,  1.00, -1.00,  1.00,  1.00,  1.00, -0.10],
        [ 1.00,  0.95, -0.87, -1.00,  0.98, -1.00,  1.00, -1.00],
        [ 0.89,  0.07, -1.00, -1.00,  1.00,  1.00, -1.00, -0.22],
        [ 1.00, -1.00, -1.00, -1.00,  0.68,  1.00, -1.00,  1.00],
        [ 1.00,  1.00, -1.00, -1.00,  0.10,  0.60,  1.00, -1.00],
        [ 1.00,  1.00, -1.00, -1.00,  1.00,  0.95, -0.30, -1.00],
        [ 0.82,  0.98, -1.00, -1.00,  0.99, -0.19,  0.99,  0.91],
        [-0.91,  1.00, -0.98, -1.00,  1.00,  1.00,  0.97,  0.97],
        [ 1.00,  1.00, -1.00, -1.00, -1.00, -1.00,  1.00, -1.00],
        [ 0.64,  1.00,  1.00, -1.00,  1.00,  1.00, -1.00, -0.82],
        [ 1.00,  1.00, -1.00, -1.00,  1.00, -0.01,  1.00, -1.00],
        [ 1.00,  1.00, -1.00, -1.00, -0.86,  1.00,  1.00, -1.00],


## Tanh


## Softmax

&ensp; &ensp; &ensp; &ensp; **Softmax** is an activation function often used in the output layer of neural networks. It transforms raw neural network outputs, known as logits, into probability distributions where each probability represents the model's confidence that a given example belongs to a specific class.

```{figure} ../images/softmax.png
---
width: 500px
name: softmax
---
Softmax
```

In [13]:
probs = logits.exp() / logits.exp().sum(1, keepdims=True)

In [14]:
print(f"probs    {list(probs.shape)}    I just printed the first 15 rows, there would be 60000!")
print(probs[:15])

probs    [60000, 10]    I just printed the first 15 rows, there would be 60000!
tensor([[    0.02,     0.10,     0.75,     0.02,     0.01,     0.01,     0.02,     0.00,     0.07,     0.00],
        [    0.04,     0.03,     0.41,     0.00,     0.28,     0.13,     0.00,     0.00,     0.09,     0.01],
        [    0.03,     0.05,     0.03,     0.00,     0.49,     0.00,     0.00,     0.00,     0.05,     0.35],
        [    0.01,     0.19,     0.11,     0.27,     0.17,     0.02,     0.03,     0.00,     0.17,     0.01],
        [    0.01,     0.03,     0.29,     0.12,     0.31,     0.13,     0.00,     0.00,     0.10,     0.01],
        [    0.00,     0.14,     0.03,     0.21,     0.19,     0.17,     0.00,     0.00,     0.21,     0.04],
        [    0.03,     0.20,     0.60,     0.03,     0.00,     0.01,     0.06,     0.01,     0.07,     0.00],
        [    0.02,     0.19,     0.48,     0.09,     0.04,     0.02,     0.03,     0.00,     0.13,     0.00],
        [    0.01,     0.03,     0.18,  

## Calculate Loss

&ensp; &ensp; &ensp; &ensp; The forward pass can also be seen as the step where the model, using its current parameters, generates predictions. The **Cross-Entropy Loss** is a widely used loss function for classification tasks that evaluates how well these predictions align with the labels. It combines Softmax with the average negative log-likelihood.

```{admonition} Likelihood

The **likelihood** represents the joint probability of the model assigning correct labels to all examples:

<br>

$$
\text{likelihood} = \prod_{i=1}^{N} p_i
$$

where:  
- $p_i$ is the probability assigned by the model to the correct class for the $i$-th example.  
- $N$ is the total number of examples in the dataset.
```

```{admonition} Log Likelihood

Since each $p_i$ is a value between 0 and 1, their product can become very small. To avoid numerical instability, we take the logarithm of the likelihood:

<br>

$$
\text{log likelihood} = \log \left(\prod_{i=1}^{N} p_i \right) = \sum_{i=1}^{N} \log(p_i)
$$
```


````{admonition} Negative Log Likelihood

Looking at the graph of the logarithmic function, please note that:

- If we pass in a probability of $1$, the log probability is $0$.
- If we pass in a lower probability $\left(0 < p < 1 \right)$, the log probability becomes more negative.  
- If we pass in a probability of 0, the log probability is $-\infty$.

```{figure} ../images/log-function.png
---
width: 250px
name: log-function
---
Logarithmic Function
```

<br>

Thus, when all the individual probabilities are 1 (the best-case scenario), the log likelihood is 0, and when the probabilities decrease, the log likelihood becomes more negative. To make it eassier to interpret, we use the **negative log likelihood (NLL)**, a positive metric where values closer to 0 indicate better predictions:

<br>

$$
\text{NLL} = - \sum_{i=1}^{N} \log(p_i)
$$

````

```{admonition} Average Negative Log Likelihood

To normalize the NLL, it is often divided by the total number of examples in the dataset:

<br>

$$
\text{Average NLL} = - \frac{1}{N} \sum_{i=1}^{N} \log(p_i)
$$

```

In [15]:
# number of train images 
N = X.shape[0]

# average NLL
loss = -probs[range(N), train_y].log().mean()

print(f"Loss: {loss:.4f}")

Loss: 3.6710


## Backward Pass

&ensp; &ensp; &ensp; &ensp; A lower loss indicates that the model is making more accurate predictions by assigning higher probabilities to the correct classes. Thus, to improve the model's performance, we aim to minimize the loss by adjusting its parameters. 

&ensp; &ensp; &ensp; &ensp; During the **backward pass**, or backpropagation, we calculate the gradient of the loss function with respect to these parameters. This gradient is determined by computing the derivative of the loss with respect to the model's parameters using the chain rule. To simplify this process, we will break the forward pass into smaller components, making it easier to differentiate.

In [16]:
# hidden layer 1
h1_pre = X @ W1 + b1
h1 = torch.tanh(h1_pre)

# hidden layer 2
h2_pre = h1 @ W2 + b2
h2 = torch.tanh(h2_pre)

# output layer
logits = h2 @ W3 + b3

# softmax
counts = logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv

# average NLL
log_probs = probs.log()
loss = -log_probs[range(N), train_y].mean()

We first calculate the derivative of the loss with respect to the log probabilities.

<br>

$$
loss = - mean(log\_probs) \quad \Rightarrow \quad
\begin{matrix}
dlog\_probs = 0 \text{ if not p_i}\\
dlog\_probs = -\frac{1}{N} \text{if p_i}
\end{matrix}
$$

In [17]:
dlog_probs = torch.zeros_like(log_probs)
dlog_probs[range(N), train_y] = -1.0 / N

Next, we calculate the derivative of the loss with respect to the probabilities using the chain rule.

<br>

$$
log\_probs = log(probs) \quad \Rightarrow \quad dprobs = \frac{1}{probs} \cdot dlog\_probs
$$

In [18]:
dprobs = (1.0 / probs) * dlog_probs

We will continue backpropagating by calculating the derivative of the loss with respect to the intermediate values, and then, with respect to the model's parameters.

<br>

$$
probs = counts \cdot counts\_sum\_inv \quad \Rightarrow \quad
\begin{matrix}
dcounts = counts\_sum\_inv \cdot dprobs \\
dcounts\_sum\_inv = counts \cdot dprobs
\end{matrix}
$$

In [19]:
dcounts = counts_sum_inv * dprobs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)

$$
counts\_sum\_inv = \frac{1}{counts\_sum} \quad \Rightarrow \quad dcounts\_sum = -\frac{1}{\sqrt{counts\_sum}} \cdot dcounts\_sum\_inv
$$

In [20]:
dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv

$$
counts\_sum = \text{sum of rows of counts} \quad \Rightarrow \quad dcounts = dcounts\_sum
$$

In [21]:
dcounts += torch.ones_like(counts) * dcounts_sum

$$
counts = e^{logits} \quad \Rightarrow \quad dlogits = counts \cdot dcounts
$$

In [22]:
dlogits = counts * dcounts

$$
\text{logits} = h2 \times W3 + b3 \quad \Rightarrow \quad
\begin{matrix}
dh2 = dlogits \times (W3)^T \\
dW3 = (h2)^T \times dlogits \\
db3 = \text{sum of columns of dlogits}
\end{matrix}
$$

In [23]:
dh2 = dlogits @ W3.T
dW3 = h2.T @ dlogits
db3 = dlogits.sum(0)

$$
h2 = tanh(h2\_pre) \quad \Rightarrow \quad dh2\_pre = (1 - (h2)^2) \cdot dh2
$$

In [24]:
dh2_pre = (1.0 - h2**2) * dh2

$$
h2\_pre = h1 \times W2 + b2 \quad \Rightarrow \quad
\begin{matrix}
dh1 = dh2\_pre \times (W2)^T \\
dW2 = (h1)^T \times dh2\_pre \\
db2 = \text{sum of columns of dh2\_pre}
\end{matrix}
$$

In [25]:
dh1 = dh2_pre @ W2.T
dW2 = h1.T @ dh2_pre
db2 = dh2_pre.sum(0)

$$
h1 = tanh(h1\_pre) \quad \Rightarrow \quad dh1\_pre = (1 - (h1)^2) \cdot dh1
$$

In [26]:
dh1_pre = (1.0 - h1**2) * dh1

$$
h1\_pre = X \times W1 + b1 \quad \Rightarrow \quad
\begin{matrix}
dh1 = dh1\_pre \times (W1)^T \\
dW1 = (X)^T \times dh1\_pre \\
db1 = \text{sum of columns of dh1\_pre}
\end{matrix}
$$

In [27]:
dX = dh1_pre @ W1.T
dW1 = X.T @ dh1_pre
db1 = dh1_pre.sum(0)

We will create a list that contains the gradient of the loss function with respect to the parameters. 

In [28]:
grads = [dW1, db1, dW2, db2, dW3, db3]

## Update Parameters

&ensp; &ensp; &ensp; &ensp; A gradient is a vector that indicates the direction and rate of the steepest increase in a function's value. To minimize the loss function, we use **gradient descent**, an optimization algorithm that updates the model's parameters by moving in the opposite direction of the gradient.

```{admonition} Gradient descent

Gradient descent updates each parameter by subtracting the gradient scaled by a factor (known as the learning rate) from its current value.

<br>

$$
\theta = \theta - \eta \cdot \nabla L(\theta)
$$

where:
- $\theta$ represents the parameters of the neural network.
- $\eta$ is the learning rate.
- $\nabla L(\theta)$ is the gradient of the loss function with respect to the parameters.
```


```{important}
The learning rate determines the size of the steps we take towards the minimum loss. A smaller learning rate results in more precise but slower convergence, while a larger learning rate can speed up convergence but may risk overshooting the minimum.
```

In [29]:
lr = 0.1 
for p, grad in zip(parameters, grads):
    p.data += -lr * grad

## Training

&ensp; &ensp; &ensp; &ensp; During **training**, a neural network iteratively performs a forward pass, calculates the loss, makes a backward pass, and updates its parameters. In this case, since the entire trainig dataset is processed in each training iteration, we can refer to each iteration as an epoch.

In [ ]:
epochs = 100       # train iterations
lr = 0.1           # learning rate

# intialize neural network
parameters = initialize_nn()
W1, b1, W2, b2, W3, b3 = parameters

for epoch in range(epochs):

    # -------------------- forward pass --------------------

    # hidden layer 1
    h1_pre = X @ W1 + b1
    h1 = torch.tanh(h1_pre)

    # hidden layer 2
    h2_pre = h1 @ W2 + b2
    h2 = torch.tanh(h2_pre)

    # output layer
    logits = h2 @ W3 + b3

    # softmax
    counts = logits.exp()
    counts_sum = counts.sum(1, keepdims=True)
    counts_sum_inv = counts_sum**-1
    probs = counts * counts_sum_inv


    # -------------------- calculate loss --------------------

    # average negative log liklihood
    log_probs = probs.log()
    loss = -log_probs[range(N), train_y].mean()

    # print loss every 10 epochs
    if epoch % 20 == 0 or epoch == epochs - 1:
        print(f"Epoch: {epoch:2d}/{epochs}     Loss: {loss.item():.4f}")
    

    # -------------------- backward pass --------------------

    dlog_probs = torch.zeros_like(log_probs)
    dlog_probs[range(N), train_y] = -1.0 / N

    dprobs = (1.0 / probs) * dlog_probs
    dcounts = counts_sum_inv * dprobs
    dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
    dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
    dcounts += torch.ones_like(counts) * dcounts_sum

    dlogits = counts * dcounts
    dh2 = dlogits @ W3.T
    dW3 = h2.T @ dlogits
    db3 = dlogits.sum(0)
    dh2_pre = (1.0 - h2**2) * dh2
    dh1 = dh2_pre @ W2.T
    dW2 = h1.T @ dh2_pre
    db2 = dh2_pre.sum(0)
    dh1_pre = (1.0 - h1**2) * dh1
    dX = dh1_pre @ W1.T
    dW1 = X.T @ dh1_pre
    db1 = dh1_pre.sum(0)

    grads = [dW1, db1, dW2, db2, dW3, db3]


    # -------------------- update parameters --------------------

    for p, grad in zip(parameters, grads):
        p.data += -lr * grad

Epoch:  0/100     Loss: 3.6710
Epoch: 10/100     Loss: 3.1006
Epoch: 20/100     Loss: 2.8051
Epoch: 30/100     Loss: 2.6102
Epoch: 40/100     Loss: 2.4914
Epoch: 50/100     Loss: 2.4206
Epoch: 60/100     Loss: 2.3751
Epoch: 70/100     Loss: 2.3430
Epoch: 80/100     Loss: 2.3183
Epoch: 90/100     Loss: 2.2988
Epoch: 99/100     Loss: 2.2843
